# Week 64, Day 2: Bytecode and dis

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w064d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
stack = []
program = [("PUSH", 2), ("PUSH", 3), ("PUSH", 4), ("MUL",), ("ADD",)]
for instruction in program:
    op = instruction[0]
    if op == "PUSH":
        stack.append(instruction[1])
    else:
        right, left = stack.pop(), stack.pop()
        stack.append(left * right if op == "MUL" else left + right)
    print(op, stack)

In [ ]:
def run_stack(program, variables):
    stack = []
    for op, *arg in program:
        if op == "PUSH":
            stack.append(arg[0])
        elif op == "LOAD":
            stack.append(variables[arg[0]])
        elif op == "STORE":
            variables[arg[0]] = stack.pop()
        else:
            right, left = stack.pop(), stack.pop()
            stack.append({"ADD": left + right, "SUB": left - right, "MUL": left * right}[op])
    return variables

prog = [("LOAD", "price"), ("PUSH", 1), ("LOAD", "tax"), ("ADD",), ("MUL",), ("STORE", "total")]
print(run_stack(prog, {"price": 100, "tax": 0.2}))

In [ ]:
import dis

def price_with_tax(price, rate):
    tax = price * rate
    return price + tax

dis.dis(price_with_tax)

names = [ins.opname for ins in dis.get_instructions(price_with_tax)]
print(len(names), "instructions; the last is", names[-1])
print(price_with_tax.__code__.co_varnames, price_with_tax.__code__.co_consts)

In [ ]:
import dis

RATE = 0.2

def with_global(prices):
    return [p * RATE for p in prices]

def kinds(func):
    found = set()
    for ins in dis.get_instructions(func):
        for family in ("LOAD_GLOBAL", "LOAD_FAST", "LOAD_ATTR", "STORE", "CALL", "RETURN"):
            if ins.opname.startswith(family):
                found.add(family)
    return sorted(found)

print(kinds(with_global))
print([ins.argval for ins in dis.get_instructions(with_global) if ins.opname.startswith("LOAD_GLOBAL")])

In [ ]:
def run(program, variables):
    stack, pc = [], 0
    while pc < len(program):
        op, *arg = program[pc]
        pc += 1
        if op == "PUSH":
            stack.append(arg[0])
        elif op == "LOAD":
            stack.append(variables[arg[0]])
        elif op == "STORE":
            variables[arg[0]] = stack.pop()
        elif op == "JUMP":
            pc = arg[0]
        elif op == "JUMP_IF_FALSE":
            if not stack.pop():
                pc = arg[0]
        else:
            right, left = stack.pop(), stack.pop()
            stack.append({"SUB": left - right, "ADD": left + right, "GT": left > right}[op])
    return variables

countdown = [("LOAD", "n"), ("PUSH", 0), ("GT",), ("JUMP_IF_FALSE", 13),     # 0 to 3: the test
             ("LOAD", "n"), ("PUSH", 1), ("SUB",), ("STORE", "n"),           # 4 to 7: n = n - 1
             ("LOAD", "steps"), ("PUSH", 1), ("ADD",), ("STORE", "steps"),   # 8 to 11: steps += 1
             ("JUMP", 0)]                                                    # 12: back to the test
print(run(countdown, {"n": 4, "steps": 0}))

In [ ]:
import dis, importlib.util, sys

def f(x):
    return x + 1

print(sys.version_info[:2], len(importlib.util.MAGIC_NUMBER), "byte magic number")
ops = [i.opname for i in dis.get_instructions(f)]
print("RETURN_VALUE" in ops or "RETURN_CONST" in ops)
print(len(f.__code__.co_code) % 2 == 0)          # instructions are two bytes each

> **Key idea.** CPython's bytecode runs on a stack machine: loads push values, stores pop them into variables, operations pop their inputs and push results, and expressions compile to post-order stack code. dis.dis prints a function's instructions with line numbers, offsets, opnames and decoded arguments; dis.get_instructions gives them as objects. Instructions come in families: loads (LOAD_FAST for locals is faster than LOAD_GLOBAL), stores, operations, calls and returns, iteration, and jumps, which implement if, while, and, or. Bytecode is an implementation detail that changes every release (and specialises itself at run time since 3.11), so use dis to understand, never as something to depend on.

### Exercise: A stack machine

Write `execute(program, variables=None)` for a stack machine whose program is a list of tuples: `("PUSH", value)`, `("LOAD", name)`, `("STORE", name)`, `("ADD",)`, `("SUB",)`, `("MUL",)`, `("DIV",)` (true division), `("NEG",)` (negate the top value), `("DUP",)` (push a copy of the top value), `("POP",)` and `("PRINT",)` (pop and append the value to an output list). Return a tuple `(variables, output)`. Popping an empty stack raises `IndexError("stack underflow")`, an unknown instruction `ValueError`, and an unknown variable `NameError`. Don't change the `variables` dict passed in.

In [ ]:
def execute(program, variables=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_vm():\n    \"a stack machine\"\n    prog = [(\"LOAD\", \"w\"), (\"LOAD\", \"h\"), (\"MUL\",), (\"DUP\",), (\"STORE\", \"area\"), (\"PUSH\", 2), (\"DIV\",), (\"PRINT\",)]\n    start = {\"w\": 3, \"h\": 4}\n    variables, out = execute(prog, start)\n    assert variables == {\"w\": 3, \"h\": 4, \"area\": 12} and out == [6.0] and start == {\"w\": 3, \"h\": 4}\n    assert execute([(\"PUSH\", 5), (\"NEG\",), (\"PUSH\", 2), (\"SUB\",), (\"PRINT\",)])[1] == [-7]\n    assert execute([(\"PUSH\", 1), (\"POP\",)]) == ({}, [])\n    for prog, err in (([(\"ADD\",)], IndexError), ([(\"JUMP\", 0)], ValueError), ([(\"LOAD\", \"x\")], NameError)):\n        try:\n            execute(prog)\n            assert False\n        except err:\n            pass")

### Exercise: Compiling expressions

Write `compile_expression(source)`: parse the expression with `ast.parse(source, mode="eval")` and return a list of instructions for the machine above that leaves its value on the stack: constants become `PUSH`, names `LOAD`, `+ - * /` become `ADD SUB MUL DIV`, and unary minus `NEG` (unary plus produces nothing). Anything else raises `ValueError`. Then `stack_depth(program)`: the largest number of values on the stack at any moment while running a program of `PUSH`, `LOAD`, binary operations and `NEG`, worked out **without** running it.

In [ ]:
import ast

def compile_expression(source):
    ___

def stack_depth(program):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_compile():\n    \"AST to stack code\"\n    assert compile_expression(\"price * (1 + tax)\") == [(\"LOAD\", \"price\"), (\"PUSH\", 1), (\"LOAD\", \"tax\"), (\"ADD\",), (\"MUL\",)]\n    assert compile_expression(\"-a - +b / 2\") == [(\"LOAD\", \"a\"), (\"NEG\",), (\"LOAD\", \"b\"), (\"PUSH\", 2), (\"DIV\",), (\"SUB\",)]\n    assert compile_expression(\"7\") == [(\"PUSH\", 7)]\n    for bad in (\"a ** 2\", \"f(x)\", \"a < b\"):\n        try:\n            compile_expression(bad)\n            assert False\n        except ValueError:\n            pass\n    assert stack_depth(compile_expression(\"a + b + c + d\")) == 2\n    assert stack_depth(compile_expression(\"a + (b + (c + d))\")) == 4\n    assert stack_depth([]) == 0")

### Exercise: Jumps and loops

Extend the machine with control flow. Write `execute_jumps(program, variables=None, limit=10000)` supporting everything in the first exercise plus comparisons `("LT",)`, `("GT",)` and `("EQ",)` (pushing `True` or `False`), `("JUMP", target)`, `("JUMP_IF_FALSE", target)` (pops the test) and `("JUMP_IF_TRUE", target)`, where targets are positions in the list. Running more than `limit` instructions raises `RuntimeError("too many steps")`. Return `(variables, output)`. Then write `sum_to_program()`: a program that, given a variable `n`, sets `total` to `1 + 2 + ... + n` using a loop.

In [ ]:
def execute_jumps(program, variables=None, limit=10000):
    ___

def sum_to_program():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_jumps():\n    \"jumps\"\n    prog = sum_to_program()\n    assert execute_jumps(prog, {\"n\": 4})[0][\"total\"] == 10\n    assert execute_jumps(prog, {\"n\": 0})[0][\"total\"] == 0\n    assert execute_jumps(prog, {\"n\": 100})[0][\"total\"] == 5050\n    pick = [(\"LOAD\", \"x\"), (\"PUSH\", 0), (\"LT\",), (\"JUMP_IF_TRUE\", 7), (\"PUSH\", \"positive\"), (\"PRINT\",), (\"JUMP\", 9),\n            (\"PUSH\", \"negative\"), (\"PRINT\",)]\n    assert execute_jumps(pick, {\"x\": 5})[1] == [\"positive\"] and execute_jumps(pick, {\"x\": -1})[1] == [\"negative\"]\n    assert execute_jumps([(\"PUSH\", 2), (\"PUSH\", 2), (\"EQ\",), (\"PRINT\",)])[1] == [True]\n    try:\n        execute_jumps([(\"JUMP\", 0)])\n        assert False\n    except RuntimeError as e:\n        assert str(e) == \"too many steps\"")

### Exercise: Asking dis

Using `dis.get_instructions` (and checking only opname **prefixes**, which are stable), write `globals_used(func)`: the sorted list of names a function loads with an instruction whose opname starts with `LOAD_GLOBAL`, and `has_loop(func)`: True if the function contains an instruction whose opname starts with `FOR_ITER` or is a backwards jump (an opname starting with `JUMP_BACKWARD`). Then `local_names(func)`: the function's local variable names, from the code object.

In [ ]:
import dis

def globals_used(func):
    ___

def has_loop(func):
    ___

def local_names(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_dis():\n    \"reading bytecode\"\n    def report(items):\n        total = 0\n        for item in items:\n            total += len(item)\n        return round(total / max(1, len(items)), 2)\n    def simple(a, b):\n        c = a + b\n        return c\n    def waiting(n):\n        while n > 0:\n            n -= 1\n        return n\n    assert globals_used(report) == [\"len\", \"max\", \"round\"] and globals_used(simple) == []\n    assert has_loop(report) and has_loop(waiting) and not has_loop(simple)\n    assert local_names(report) == [\"items\", \"total\", \"item\"] and local_names(simple) == [\"a\", \"b\", \"c\"]")

### Exercise: A peephole optimiser

Compilers improve code by looking at short windows of instructions, a **peephole**. Write `optimise(program)` for the first exercise's machine, repeating until nothing changes: replace `PUSH a, PUSH b, <op>` with `PUSH result` for `ADD`, `SUB` and `MUL`; replace `PUSH a, NEG` with `PUSH -a`; remove `PUSH x, POP` pairs; and replace `STORE n, LOAD n` with `DUP, STORE n`. Return the new list (don't change the original).

In [ ]:
def optimise(program):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_peephole():\n    \"peephole optimisation\"\n    prog = [(\"PUSH\", 2), (\"PUSH\", 3), (\"MUL\",), (\"PUSH\", 1), (\"ADD\",), (\"STORE\", \"x\"), (\"LOAD\", \"x\"), (\"PRINT\",)]\n    assert optimise(prog) == [(\"PUSH\", 7), (\"DUP\",), (\"STORE\", \"x\"), (\"PRINT\",)]\n    assert prog[0] == (\"PUSH\", 2)\n    assert optimise([(\"PUSH\", 4), (\"NEG\",), (\"PUSH\", 9), (\"POP\",), (\"LOAD\", \"a\"), (\"ADD\",)]) == [(\"PUSH\", -4), (\"LOAD\", \"a\"), (\"ADD\",)]\n    assert optimise([(\"LOAD\", \"a\"), (\"PUSH\", 1), (\"ADD\",)]) == [(\"LOAD\", \"a\"), (\"PUSH\", 1), (\"ADD\",)]\n    assert optimise([]) == []")

### Bug hunt: Operands the wrong way round

The machine pops the **right** operand first (it was pushed last), but this code calls the first value it pops `left`, so `10 - 4` comes out as `-6`. Pop in the right order so this prints `6`.

In [ ]:
stack = []
for op, *arg in [("PUSH", 10), ("PUSH", 4), ("SUB",)]:
    if op == "PUSH":
        stack.append(arg[0])
    elif op == "SUB":
        left, right = stack.pop(), stack.pop()
        stack.append(left - right)
print(stack[-1])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"pop order\"\n    expect_output(\"6\")")

### Bug hunt: A jump that never happens

After a jump, the loop still adds one to the program counter, so every jump lands one instruction too far: the jump to position 2 ends up at position 3, so `PUSH "kept"` never runs and this prints `[]`. Advance the counter **before** carrying out the instruction, so a jump replaces it, and this prints `['kept']`.

In [ ]:
program = [("JUMP", 2), ("PUSH", "skipped"), ("PUSH", "kept"), ("PRINT",)]
stack, out, pc = [], [], 0
while pc < len(program):
    op, *arg = program[pc]
    if op == "JUMP":
        pc = arg[0]
    elif op == "PUSH":
        stack.append(arg[0])
    elif op == "PRINT":
        out.append(stack.pop()) if stack else None
    pc += 1
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the program counter\"\n    expect_output(\"['kept']\")")

### Bug hunt: Operations before operands

The compiler emits the operation **before** its operands (pre-order), so the machine tries to add with nothing on the stack and crashes with `IndexError`. Emit the operands first (post-order) so this prints `14`.

In [ ]:
import ast

def emit(node, code):
    if isinstance(node, ast.Constant):
        code.append(("PUSH", node.value))
    elif isinstance(node, ast.BinOp):
        code.append(("ADD",) if isinstance(node.op, ast.Add) else ("MUL",))
        emit(node.left, code)
        emit(node.right, code)

code = []
emit(ast.parse("2 + 3 * 4", mode="eval").body, code)
stack = []
for op, *arg in code:
    if op == "PUSH":
        stack.append(arg[0])
    else:
        b, a = stack.pop(), stack.pop()
        stack.append(a + b if op == "ADD" else a * b)
print(stack[-1])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"post-order\"\n    expect_output(\"14\")")

### Puzzle: Emitting stack code

Arrange the lines of a function that emits stack code for additions of names. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the stack after PUSH 1, PUSH 2, PUSH 3, ADD, as a list
q = ___   # True or False: LOAD_FAST looks a local up by name in a dict
r = ___   # which statement has no instruction of its own and becomes jumps: "if" or "return"
s = ___   # True or False: the same function always has the same bytecode in every Python version
t = ___   # the largest stack depth of stack code for a + b * c

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 5] and q is False and r == \"if\", \"p, q and r\"\n    assert s is False and t == 3, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: compiling a little language

Write `compile_program(source)`, which compiles a small subset of Python into instructions for your `execute_jumps` machine, and `run_program(source, variables=None)`, which compiles and runs it, returning `(variables, output)`. Copy `execute_jumps` into your answer. Support:

- expressions: constants, names, `+ - *`, `/`, unary `-`, comparisons `<`, `>`, `==` (single comparisons only), and `and` / `or` with short-circuiting (the result is one of the operands, as in Python; hint: `DUP` and `POP` help);
- statements: assignment to one name, augmented assignment (`x += e`, `x -= e`, `x *= e`), `print(e)` with one argument (the `PRINT` instruction), `if` / `else` (including `elif`, which the AST represents as a nested `If`), and `while` loops (without `else`).

Anything else raises `ValueError`. Jump targets must be correct positions in the final list: emit a placeholder and **patch** it once you know where the jump should go.

In [ ]:
import ast

def execute_jumps(program, variables=None, limit=10000):
    ___

def compile_program(source):
    ___

def run_program(source, variables=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_programs():\n    \"a compiled language\"\n    src = \"total = 0\\ni = 1\\nwhile i < n + 1:\\n    total += i\\n    i += 1\\nprint(total)\\n\"\n    assert run_program(src, {\"n\": 10})[1] == [55]\n    fizz = (\n        \"i = 1\\nwhile i < 16:\\n\"\n        \"    if i - (i / 15) * 15 == 0 and 0:\\n        print(0)\\n\"\n        \"    elif i == 15:\\n        print('fizzbuzz')\\n\"\n        \"    elif i == 3 or i == 6:\\n        print('fizz')\\n\"\n        \"    else:\\n        x = 1\\n\"\n        \"    i += 1\\n\"\n    )\n    assert run_program(fizz)[1] == [\"fizz\", \"fizz\", \"fizzbuzz\"]\n    assert run_program(\"a = 0 or 5\\nb = 3 and 0\\nc = 2 and 7\\nprint(a)\\nprint(b)\\nprint(c)\")[1] == [5, 0, 7]\n    assert run_program(\"x = -3 * 2\\nx -= 1\\nx *= 2\\nprint(x / 4)\")[1] == [-3.5]\n    assert run_program(\"if 1 > 2:\\n    print('no')\\nprint('done')\")[1] == [\"done\"]\n\ndef test_refused():\n    \"unsupported code\"\n    for bad in (\"for i in range(3):\\n    pass\", \"def f():\\n    pass\", \"x = [1]\", \"print(1, 2)\", \"a, b = 1, 2\", \"x = 1 < 2 < 3\"):\n        try:\n            compile_program(bad)\n            assert False, bad\n        except ValueError:\n            pass")

## Warm-up

Five instructions computed `2 + 3 * 4`: push the operands, and let each operation take the top two values and push its result. CPython runs your code in exactly this style, with a few hundred kinds of instruction instead of three.

## A stack machine

A **stack machine** keeps its working values on a stack instead of in named registers. Every instruction either **pushes** something (a constant, the value of a variable), **pops** something (to store it in a variable, to discard it, to return it), or **pops its inputs and pushes its result** (arithmetic, comparisons, calls).

To turn an expression into stack code, walk its syntax tree **post-order**: emit code for the operands first, then the operation. For `price * (1 + tax)`:

```
LOAD price
PUSH 1
LOAD tax
ADD          # pops 1 and tax, pushes 1 + tax
MUL          # pops price and (1 + tax), pushes the product
```

Stack code is compact (no register numbers to encode), simple to generate from a tree, and simple to interpret, which is why CPython, the Java virtual machine, WebAssembly and many others use it.

## Reading dis

`dis.dis(func)` prints the bytecode of a function (or of a string of code). Each line shows:

- the **source line number** (at the start of each line's instructions);
- the instruction's **offset** in bytes within `co_code` (each instruction takes two bytes, plus extra "cache" entries for some);
- the **opname**, such as `LOAD_FAST` or `BINARY_OP`;
- its **argument** (a number) and, in brackets, what that number means: the local variable, constant, name or operator it refers to.

`dis.get_instructions(func)` gives the same information as `Instruction` objects with `opname`, `arg`, `argval` (the decoded argument), `offset` and position fields, so tools can read bytecode without parsing text. `dis.show_code(func)` prints the code object's constants and names.

The output below is from whichever Python is running this page; don't be surprised if it differs from what you see on your own computer.

## Families of instructions

The exact names change between versions, but the families are stable:

- **Loads** push a value: `LOAD_CONST` (from `co_consts`), `LOAD_FAST` (a local, by index into the frame's local slots: fast because no dict lookup is needed), `LOAD_GLOBAL` (a global or built-in, looked up by name), `LOAD_ATTR` (an attribute of the top value), `LOAD_DEREF` (a closure cell).
- **Stores** pop a value into a place: `STORE_FAST`, `STORE_GLOBAL`, `STORE_ATTR`, `STORE_SUBSCR`.
- **Operations**: `BINARY_OP` (with the operator as its argument, since 3.11), `UNARY_NEGATIVE`, `COMPARE_OP`, `CONTAINS_OP`, `IS_OP`.
- **Calls and returns**: `CALL`, `RETURN_VALUE`, and building containers with `BUILD_LIST`, `BUILD_TUPLE`, `BUILD_MAP`.
- **Jumps**: conditional and unconditional jumps change which instruction runs next (next section).
- **Iteration**: `GET_ITER` and `FOR_ITER` implement `for` loops using Week 59's iterator protocol.

Seeing these explains some performance folklore: a local variable is faster than a global because `LOAD_FAST` indexes an array while `LOAD_GLOBAL` searches dicts; `x.y.z` in a loop repeats two `LOAD_ATTR`s every time; and a list comprehension avoids the `.append` lookup and call that a loop does.

## Jumps: how if and while work

Bytecode has no `if` or `while`: they become **jumps**. A conditional jump pops the top of the stack and, depending on its truth, either carries on to the next instruction or jumps to another offset. An `if` jumps **over** its body when the test is false; a `while` loop tests, jumps out when false, and ends its body with a jump **back** to the test. `and` and `or` are jumps too, which is how short-circuiting works: `a and b` jumps past `b` when `a` is false, leaving `a` as the result.

In the stack machine you'll build, instructions are numbered by position, and a jump's argument is the position to go to:

```
0  LOAD n
1  PUSH 0
2  GT                  # n > 0 ?
3  JUMP_IF_FALSE 9     # no: leave the loop
4  LOAD n
5  PUSH 1
6  SUB
7  STORE n             # n = n - 1
8  JUMP 0              # back to the test
9  ...
```

## Bytecode changes between versions

Bytecode is an **implementation detail** of CPython, not part of the Python language. It changes in almost every release: Python 3.11 merged the arithmetic instructions into `BINARY_OP` and added **adaptive specialisation** (an instruction such as `BINARY_OP` rewrites itself into a faster version like `BINARY_OP_ADD_INT` after seeing the same types a few times), 3.12 and 3.13 combined common pairs of instructions, and 3.14 changed how small integers are loaded. Other Pythons (PyPy, MicroPython) use completely different internal forms.

So:

- `.pyc` files are tied to one version (yesterday), and each has a **magic number** in its header (`importlib.util.MAGIC_NUMBER`) that changes with the bytecode;
- tools that work on bytecode (debuggers, coverage tools, some decompilers) must be updated for each release;
- your code should never depend on particular instructions. Use `dis` to **understand**, and measure performance with timers rather than counting instructions (Week 65).

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, run `python -m dis your_file.py` on a short program, then on the same file with a different Python version if you have one, and compare. Tomorrow: frames, the objects that hold each running function's state, and the evaluation loop that drives them.